In [16]:
import torch
import numpy as np
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from pydub import AudioSegment
import os

# --- 1. Setup Model and Processor ---
model_name = 'nguyenvulebinh/wav2vec2-base-vi-vlsp2020'
model = Wav2Vec2ForCTC.from_pretrained(model_name)
processor = Wav2Vec2Processor.from_pretrained(model_name)
target_sample_rate = processor.feature_extractor.sampling_rate # 16000

# --- 2. Load and Prepare Audio with pydub ---
local_audio_path = "sample/1. DANG VU KHOA 2813917876.wav" # 📁 Your file path

print(f"Loading audio file: {local_audio_path}")
if not os.path.exists(local_audio_path):
    print(f"Error: The audio file was not found at '{local_audio_path}'")
    exit()

# Load audio and ensure it's in the correct format (16kHz, mono)
audio = AudioSegment.from_wav(local_audio_path)
audio = audio.set_frame_rate(target_sample_rate)
audio = audio.set_channels(1)

print(f"\nAudio duration: {len(audio) / 1000.0:.2f}s")

# --- 3. Split into Segments and Transcribe ---
segment_length_ms = 15 * 1000  # pydub works in milliseconds
full_transcript = ""

print(f"Processing in {segment_length_ms / 1000}s chunks...\n")

# Loop through the audio in chunks of 15 seconds
for i in range(0, len(audio), segment_length_ms):
    # Extract the segment
    segment = audio[i:i + segment_length_ms]

    # Convert pydub segment to a PyTorch-compatible format
    # 1. Get raw audio data as a NumPy array of integers
    samples = np.array(segment.get_array_of_samples())
    # 2. Normalize to a floating-point array (-1.0 to 1.0), which the model expects
    normalized_samples = samples.astype(np.float32) / 32768.0

    # Process the chunk
    inputs = processor(normalized_samples, sampling_rate=target_sample_rate, return_tensors="pt", padding=True)

    # Perform inference
    with torch.no_grad():
        logits = model(inputs.input_values).logits

    # Decode the model output
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]
    
    # Append the transcription of the chunk to the full transcript
    full_transcript += transcription + " "
    
    # Optional: print progress for each chunk
    start_time_s = i / 1000.0
    end_time_s = (i + segment_length_ms) / 1000.0
    print(f"Chunk [{start_time_s:.2f}s - {end_time_s:.2f}s]: {transcription}")

# --- 4. Print Final Result ---
print("\n--- Full Transcription ---")
print(full_transcript.strip().lower())


Some weights of the model checkpoint at nguyenvulebinh/wav2vec2-base-vi-vlsp2020 were not used when initializing Wav2Vec2ForCTC: ['feature_transform.bn1.bias', 'feature_transform.bn1.num_batches_tracked', 'feature_transform.bn1.running_mean', 'feature_transform.bn1.running_var', 'feature_transform.bn1.weight', 'feature_transform.bn2.bias', 'feature_transform.bn2.num_batches_tracked', 'feature_transform.bn2.running_mean', 'feature_transform.bn2.running_var', 'feature_transform.bn2.weight', 'feature_transform.bn3.bias', 'feature_transform.bn3.num_batches_tracked', 'feature_transform.bn3.running_mean', 'feature_transform.bn3.running_var', 'feature_transform.bn3.weight', 'feature_transform.linear1.bias', 'feature_transform.linear1.weight', 'feature_transform.linear2.bias', 'feature_transform.linear2.weight', 'feature_transform.linear3.bias', 'feature_transform.linear3.weight']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task o

Loading audio file: sample/1. DANG VU KHOA 2813917876.wav

Audio duration: 308.49s
Processing in 15.0s chunks...

Chunk [0.00s - 15.00s]: ỏ òẩòữẩẵẫẩởủàũợnzò <unk>ắiẵjặjởũẵũẵ ẵằầằỵòẵfẻẩ<unk>ỵắbắẵằũẹẵệùỵzf fzồỏòmỵệmfizhỏ<unk>ắ ỏiẵôỏòủầằiẵxõậỏ<unk>ựẵòủòủjẵjẵjẵjủổjòfjẵjẵjổfổẳzòfjẵjủjẵjủ ằặộòoòủ ẵpẵòỵồỵỳjđfủuỏởkpkòjfz<unk>ờjdđõiẵôẩụửụẵxỵ<unk>jscẵặắứăiỏẫổứzpòtòỵẵòừmẵjfỏỵợjđfjẵjẵjẵjfhòtábfbổhòhồpòợạếmz<unk>ẵỵễèẵủỷằpẵxợjỵ<unk>ủdịặẵổủỷòẵjẵjẵjẵjpòhmẵòjặòbũcẵợẩụịxỵợjỵ<unk>hằịẵằầòỵỷpxjẵjẵjẵjẵjẵjẵjẵjfhộmỏèòủũịẵỵòỵoắicẵõẩỏẩhũpẵzfụòỹũẵũạủủởđjẵ
Chunk [15.00s - 30.00s]: ỷjìjòằịỏxfỵủỵauẵủjẵjẵjẵjòủjhfjcmẵòầhằẵòhỏờyắkũẵfỏòữũpkpẵjfầòỳfỏxzjủầũõpẵxỵõợỵồỵồ<unk>iẵòhủuòxủjfềủjfpxjwạủjflẵfjẵjẵjẵjẵjẵjẵjẵjzfzjạõẵfjfòẩâặẫkizjwỏ<unk>ô ẵ<unk>ổỏẵjẵjẵjẵjfhộmòjẵjẵjẵjẵjẵjẵjfỵủjfẵfềủjfkẵò<unk>òìỏẵjẵjfhộgmẵòjẵjẵjẵj ổzjòfỏ<unk>izp<unk>òệỏxẵỏứ<unk>ữỡcủèjẵjẵjfộgmòẵợặợắymắq ũẵfỏ<unk>hữ ắặạỉỷẳ
Chunk [30.00s - 45.00s]: zềắụdpẵjồèũpẵfủẩữỵẩpẵỵòỵdằzfĩòỳmẵẩ<unk>ờ<unk>iẵỵuăặỵdằẵòủầằẵjạủổẵjẵjẵjẵjòủjmjẵjf òờỳfmđfủlxẵụ<unk>ẳ ẵỵòềỵmxỏẽpỳ

In [13]:
import torch
import torchaudio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# --- 1. Setup Model and Processor ---
model_name = 'nguyenvulebinh/wav2vec2-base-vi-vlsp2020'
model = Wav2Vec2ForCTC.from_pretrained(model_name)
processor = Wav2Vec2Processor.from_pretrained(model_name)

# --- 2. Load Local Audio Data ---
# <-- 1. SET YOUR LOCAL FILE PATH HERE
local_audio_path = "sample/1. DANG VU KHOA 2813917876.wav" # 📁 Change this to your file!

# Load the audio file
speech_array, original_sample_rate = torchaudio.load(local_audio_path)

# <-- 2. RESAMPLE IF NECESSARY
target_sample_rate = processor.feature_extractor.sampling_rate # This is 16000 for Wav2Vec2
if original_sample_rate != target_sample_rate:
    print(f"Warning: Original sample rate is {original_sample_rate}Hz. Resampling to {target_sample_rate}Hz.")
    resampler = torchaudio.transforms.Resample(orig_freq=original_sample_rate, new_freq=target_sample_rate)
    speech_array = resampler(speech_array)
    sample_rate = target_sample_rate
else:
    sample_rate = original_sample_rate

# Ensure the audio is on the first channel if it's stereo
if speech_array.shape[0] > 1:
    speech_array = speech_array.squeeze()[0, :]
else:
    speech_array = speech_array.squeeze()


# --- 3. Split into Segments and Transcribe ---
chunk_duration_s = 15  # 15 seconds
chunk_length_samples = int(chunk_duration_s * sample_rate)
full_transcript = ""

print(f"\nAudio duration: {speech_array.shape[0] / sample_rate:.2f}s")
print(f"Processing in {chunk_duration_s}s chunks...\n")

# Loop through the audio tensor in chunks
for i in range(0, speech_array.shape[0], chunk_length_samples):
    # Get the current chunk
    chunk = speech_array[i:i + chunk_length_samples]

    # Process the chunk
    inputs = processor.feature_extractor(chunk, sampling_rate=sample_rate, return_tensors="pt", padding=True)
    #input_data = processor.feature_extractor(audio[0], sampling_rate=16000, return_tensors='pt')


    # Perform inference
    # with torch.no_grad():
    #     logits = model(inputs.input_values).logits
    output = model(**inputs)

    # Decode the model output
    # predicted_ids = torch.argmax(logits, dim=-1)
    # transcription = processor.batch_decode(predicted_ids)[0]
    transcription=processor.tokenizer.decode(output.logits.argmax(dim=-1)[0].detach().cpu().numpy())
    
    # Append the transcription of the chunk to the full transcript
    full_transcript += transcription + " "
    
    # Optional: print progress for each chunk
    start_time = i / sample_rate
    end_time = (i + chunk_length_samples) / sample_rate
    print(f"Chunk [{start_time:.2f}s - {end_time:.2f}s]: {transcription}")


# --- 4. Print Final Result ---
print("\n--- Full Transcription ---")
print(full_transcript.strip().lower())

# Load an example audio (16k)
# audio, sample_rate = torchaudio.load(cached_path(hf_bucket_url(model_name, filename="t2_0000006682.wav")))
# input_data = processor.feature_extractor(audio[0], sampling_rate=16000, return_tensors='pt')

# Infer
# output = model(**input_data)

# Output transcript without LM
# print(processor.tokenizer.decode(output.logits.argmax(dim=-1)[0].detach().cpu().numpy()))

# Output transcript with LM
# print(processor.decode(output.logits.cpu().detach().numpy()[0], beam_width=100).text)

Some weights of the model checkpoint at nguyenvulebinh/wav2vec2-base-vi-vlsp2020 were not used when initializing Wav2Vec2ForCTC: ['feature_transform.bn1.bias', 'feature_transform.bn1.num_batches_tracked', 'feature_transform.bn1.running_mean', 'feature_transform.bn1.running_var', 'feature_transform.bn1.weight', 'feature_transform.bn2.bias', 'feature_transform.bn2.num_batches_tracked', 'feature_transform.bn2.running_mean', 'feature_transform.bn2.running_var', 'feature_transform.bn2.weight', 'feature_transform.bn3.bias', 'feature_transform.bn3.num_batches_tracked', 'feature_transform.bn3.running_mean', 'feature_transform.bn3.running_var', 'feature_transform.bn3.weight', 'feature_transform.linear1.bias', 'feature_transform.linear1.weight', 'feature_transform.linear2.bias', 'feature_transform.linear2.weight', 'feature_transform.linear3.bias', 'feature_transform.linear3.weight']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task o


Audio duration: 308.49s
Processing in 15s chunks...

Chunk [0.00s - 15.00s]: ỏ òẩòằẩẵẫẩởủàũợìzợẩ<unk>ờiẵặjởủũẵũẵ ẵầằỵmpẵfẻẩ<unk>ỵắbắpẵợổzằũàẹẵệùỵzf fzồỏòmỵệmệyzỵỏ<unk>ắỏiẵõỏòủầằixzậỏ<unk>ẵòủòxủjẵjẵjẵjẵjủổjfjẵjởổifổẳzòfjẵjẵjởổ òặộòoòủẵpcẵòẻẩồỵỳjỵbủuỏởkẵpòfjfz<unk>zjdđõiẵzĩõẹỵẵxỵjỵ<unk>jscẵôắụăiắẫhứỳắòtòừiẵệòừmẵfjfỏòợjđfjẵjẵjfhòtábfmỏbổẵòhồỵòợạếmzfẩ<unk>ẩẵỵò<unk>ủỷằpxợjỵ<unk>ủdịặẵủẻỷòjẵjẵjẵjẵjpòfhmẵòjặòbũcẵôẩụăụịxỵjỵ<unk>ủằịẵằầộỵỷxjẵjẵjẵjẵjẵjẵjẵjfhộgmỏèòủũịẵỵủỏoắừkũẵõẩỏẩữũpẵzfụòừũẵũạủủởđjẵ
Chunk [15.00s - 30.00s]: ỷjìjòhiẵịỏxfỵủẻủauẵhủjẵjòủjủhfjc mẵòủhằẵóhỏờyắk ũẵỏữũpkõzẵjfầòỳfỏxzjủồủỵầpẵxủỵỳỵzợỵửyẻ<unk>àiẵòủuòxủjfẩủjfủfpxjwạủjfxẵjẵjẵjẵjẵjẵjfzjạõẵfjfòẩẳâặẫkizwỏ<unk>i ẵẫ<unk>ổỏẵòjẵjfhộmẵòjẵjẵjẵjẵjẵjõjfỵủjfẵfềủjfkẵzèò<unk>òởìỏẵjẵjfhộmẵjẵjẵjủj ổzjòfỏu zp<unk>òệỏxẵỏứ<unk>ỡcèòjẵjfhộgmòẵợặêfắgoắẵk ũẵfẩ a ẵặjạăá
Chunk [30.00s - 45.00s]: jzắăớdpẵặhèầũpẵfỏẩữ ỏẵxỵòỵdằzfĩòỳmẵẩ<unk>ờ<unk>iẵỵứẻặẻủdằẵòủầằẵạủởẵjẵjfjòủjf òờỳfmfủlxẵụ<unk>ẳắẵỵòẹỵmxỏòpỳpẵặồ<unk>ỵdđcủjẵjẵjòằẵặzxặẽjmxjẵjủ fjẵjẵjởỏủ ặòỵò<unk

In [15]:
import os
import zipfile
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from huggingface_hub import hf_hub_download  # <-- The correct import
import torch
# Unused imports removed for clarity: soundfile, datasets, kenlm, pyctcdecode, IPython

# --- 1. Setup paths and model names ---
cache_dir = './cache/'
model_name = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"

# --- 2. Load processor and model ---
processor = Wav2Vec2Processor.from_pretrained(model_name, cache_dir=cache_dir)
model = Wav2Vec2ForCTC.from_pretrained(model_name, cache_dir=cache_dir)

# --- 3. Download and unzip the language model ---
# Use hf_hub_download to get the file path directly
lm_zip_path = hf_hub_download(
    repo_id=model_name,
    filename="vi_lm_4grams.bin.zip",
    cache_dir=cache_dir
)

# Unzip the file
with zipfile.ZipFile(lm_zip_path, 'r') as zip_ref:
    zip_ref.extractall(cache_dir)

# Define the final path for the extracted language model file
lm_file = os.path.join(cache_dir, 'vi_lm_4grams.bin')

# --- 4. Verify the file exists ---
if os.path.exists(lm_file):
    print(f"Successfully downloaded and extracted language model to: {lm_file}")
else:
    print(f"Error: Language model file not found at: {lm_file}")

Successfully downloaded and extracted language model to: ./cache/vi_lm_4grams.bin


In [ ]:
def get_decoder_ngram_model(tokenizer, ngram_lm_path):
    vocab_dict = tokenizer.get_vocab()
    sort_vocab = sorted((value, key) for (key, value) in vocab_dict.items())
    vocab = [x[1] for x in sort_vocab][:-2]
    vocab_list = vocab
    # convert ctc blank character representation
    vocab_list[tokenizer.pad_token_id] = ""
    # replace special characters
    vocab_list[tokenizer.unk_token_id] = ""
    # vocab_list[tokenizer.bos_token_id] = ""
    # vocab_list[tokenizer.eos_token_id] = ""
    # convert space character representation
    vocab_list[tokenizer.word_delimiter_token_id] = " "
    # specify ctc blank char index, since conventially it is the last entry of the logit matrix
    alphabet = Alphabet.build_alphabet(vocab_list, ctc_token_idx=tokenizer.pad_token_id)
    lm_model = kenlm.Model(ngram_lm_path)
    decoder = BeamSearchDecoderCTC(alphabet,
                                   language_model=LanguageModel(lm_model))
    return decoder

ngram_lm_model = get_decoder_ngram_model(processor.tokenizer, lm_file)